In [17]:
import numpy as np
import torch
import tqdm

from pathlib import Path

from src.manifolds.sn_mfld import HypersphereManifold
from src.episodes.traj import generate_trajectory
from src.episodes.episode import Episode

r = np.random.default_rng(42)  # for reproducibility

In [18]:
HYPERSPHERE_MAX_DIM = 3
HYPERSPHERE_RADIUS = 1.0

NUM_TRAJECTORIES_PER_DIM = 1
NUM_INITIAL_POINTS_PER_TRAJECTORY = 11

TRAJECTORY_INITIAL_COORD_DIST = (0.0, np.pi / 4)
TRAJECTORY_WAYPOINT_INTRINSIC_COORD_DIST = (0.0, np.pi / 4)
TRAJECTORY_WAYPOINT_TRANSITION_TIME_DIST = (5.0, 1.0)

TRAJECTORY_NUM_WAYPOINTS = 5
TRAJECTORY_SAMPLE_TIME = 0.1

EPISODE_INITIAL_POS_STD = 0.4
EPISODE_INITIAL_VEL_DIST = (0, 0.1)

EPISODE_DATA_DIR = Path("../data/episodes/hypersphere")

In [19]:
episodes = []
for dim in tqdm.tqdm(range(1, HYPERSPHERE_MAX_DIM + 1), desc="Hypersphere dim", total=HYPERSPHERE_MAX_DIM):
    # constructs the intrinsic coordinate distributions in higher dimensions (non-coupled covariance)
    hd_traj_initial_pos_dist = (TRAJECTORY_INITIAL_COORD_DIST[0] * np.ones((dim,)),
                                TRAJECTORY_INITIAL_COORD_DIST[1] ** 2 * np.eye(dim))
    hd_traj_waypoint_pos_dist = (TRAJECTORY_WAYPOINT_INTRINSIC_COORD_DIST[0] * np.ones((dim,)),
                                 TRAJECTORY_WAYPOINT_INTRINSIC_COORD_DIST[1] ** 2 * np.eye(dim))
    hd_episode_initial_pos_covar = EPISODE_INITIAL_POS_STD ** 2 * np.eye(dim)
    hd_episode_initial_vel_dist = (EPISODE_INITIAL_VEL_DIST[0] * np.ones((dim,)),
                                   EPISODE_INITIAL_VEL_DIST[1] ** 2 * np.eye(dim))

    hypersphere_manifold = HypersphereManifold(dim, radius=HYPERSPHERE_RADIUS)

    for traj_idx in range(NUM_TRAJECTORIES_PER_DIM):
        traj_initial_pos = r.multivariate_normal(*hd_traj_initial_pos_dist)

        traj = generate_trajectory(start=traj_initial_pos,
                                   waypoint_dist=hd_traj_waypoint_pos_dist,
                                   waypoint_dur_dist=TRAJECTORY_WAYPOINT_TRANSITION_TIME_DIST,
                                   num_waypoints=TRAJECTORY_NUM_WAYPOINTS,
                                   dt=TRAJECTORY_SAMPLE_TIME,
                                   r=r,
                                   coord_sys=hypersphere_manifold, )

        for start_idx in range(NUM_INITIAL_POINTS_PER_TRAJECTORY):
            episode_initial_pos = hypersphere_manifold.endomorphism(
                hypersphere_manifold.default_chart,
                torch.tensor(r.multivariate_normal(
                    traj.intrinsic[hypersphere_manifold.default_chart][0, :],
                    hd_episode_initial_pos_covar))).numpy()
            episode_initial_vel = r.multivariate_normal(*hd_episode_initial_vel_dist)

            episode = Episode(traj, episode_initial_pos, episode_initial_vel, params={
                "hs_radius": HYPERSPHERE_RADIUS,
                "hs_dim": dim,
                "sample_time": TRAJECTORY_SAMPLE_TIME,
                "traj_idx": traj_idx,
                "episode_idx": start_idx,
            })
            episodes.append(
                (f"hs_dim_{dim}_radius_{HYPERSPHERE_RADIUS}_traj_{traj_idx}_episode_{start_idx}".replace(".", "_"),
                 episode))

Hypersphere dim: 100%|██████████| 3/3 [00:00<00:00, 16.45it/s]


In [20]:
for name, episode in tqdm.tqdm(episodes, desc="Episodes", total=len(episodes)):
    episode.save(EPISODE_DATA_DIR / f"{name}.npz")

Episodes: 100%|██████████| 33/33 [00:00<00:00, 4480.80it/s]
